In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {'role': 'user', 'content': text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {'role': 'assistant', 'content':text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        'model': model,
        'max_tokens': 1000,
        'messages': messages,
        'temperature': temperature,
        'stop_sequences': stop_sequences,
    }
    if system:
        params['system'] = system
    message = client.messages.create(**params)

    return message.content[0].text

In [25]:
import json

def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts 
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, 
each representing a task that required Python, JSON, or a Regex to complete.

Additionally, add solution criteria to each test case that will provide information on what a good solution would look like.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex",
        "solution_criteria": "Key criteria for evaluating the solution."
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code.

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

# Exercise: improve the score with a bettery prompt
# hint: try asking for solution criteria to be included in each test case

In [26]:
dataset = generate_dataset()

with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

In [33]:
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria you should use to evaluate the solution:
<critera>
{test_case["solution_criteria"]}
</criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this 
specific order:
- "strengths": an array of 1-3 key strengths
- "weaknesses": an array of 1-3 key areas for improvement
- "reasoning": a concise explanation of your overall assesment
- "score": a number between 1-10

Responde with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
"""
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [34]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case['task']}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
"""
    
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

In [35]:
# functions to validate the output structure
import re
import ast

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0
    
def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0
    
def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0
    
def grade_syntax(response, test_case):
    format = test_case['format']
    if format == 'json':
        return validate_json(response)
    elif format == 'python':
        return validate_python(response)
    else:
        return validate_regex(response)

In [36]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    model_score = model_grade['score']
    reasoning = model_grade['reasoning']
    
    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        'output': output,
        'test_case': test_case,
        'score': score,
        'reasoning': reasoning,
    }

In [37]:
from statistics import mean

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result['score'] for result in results])
    print(f"Average score: {average_score}")

    return results

In [38]:
with open('dataset.json', 'r') as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 6.833333333333333


In [39]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\nimport boto3\nimport re\n\ndef extract_region_from_s3_uri(uri):\n    # Parse the S3 URI\n    match = re.match(r's3://([^/]+)/?', uri)\n    if not match:\n        return None\n    \n    bucket_name = match.group(1)\n    \n    # Get the region of the bucket\n    s3_client = boto3.client('s3')\n    try:\n        response = s3_client.get_bucket_location(Bucket=bucket_name)\n        region = response.get('LocationConstraint')\n        # LocationConstraint returns None for us-east-1\n        return region if region else 'us-east-1'\n    except Exception:\n        return None\n",
    "test_case": {
      "task": "Extract the AWS region from an S3 bucket URI in the format 's3://bucket-name/key' and return the region code",
      "format": "python",
      "solution_criteria": "Function should use boto3 or AWS SDK to determine region; handle cases where region cannot be determined; return region code as string (e.g., 'us-east-1'); handle invalid URIs gracefully"
    },
   